# RAG Pipeline — ร้านฟ้าใหม่ (FahMai) — **Hybrid Search Edition**
In-Memory Retrieval-Augmented Generation for Multiple Choice QA

---
**Pipeline Overview:**
1. Ingest `.md` knowledge base files into RAM
2. Vectorize documents with `Qwen/Qwen3-Embedding-0.6B` (dense)
3. Build **BM25 sparse index** from the same chunks (sparse)
4. Load `questions.csv` (id, question, choice_1…choice_10)
5. **Hybrid Retrieval**: Dense cosine-sim ⊕ BM25 fused via **Reciprocal Rank Fusion (RRF)**
6. LLM generation via Ollama API
7. Fill `sample_submission.csv` template (id, ans) and save to `submission/`


## Cell 1 — Configuration Constants

In [45]:
# ============================================================
#  Configuration Constants
# ============================================================

KNOWLEDGE_BASE_DIR     = "../data/knowledge_base"
QUESTIONS_CSV_PATH     = "../data/questions.csv"
SAMPLE_SUBMISSION_PATH = "../data/sample_submission.csv"
SUBMISSION_DIR         = "../submission"
TRACE_LOG_PATH         = "./trace/trace_log.json"

OLLAMA_API_URL         = "http://localhost:11434/api/generate"
LLM_MODEL_NAME         = "hf.co/nectec/pathumma-thaillm-8b-think-3.0.0-GGUF:Q8_0"
EMBEDDING_MODEL_NAME   = "Qwen/Qwen3-Embedding-0.6B"
TOP_K_RETRIEVAL        = 3

# ── Hybrid Search config ──────────────────────────────────────────────────
# k1 / b  : BM25 term-frequency / document-length normalisation parameters
# RRF_K   : smoothing constant for Reciprocal Rank Fusion  (typical: 60)
# DENSE_W / SPARSE_W : weight given to each retriever in the fused score
BM25_K1   = 1.5
BM25_B    = 0.75
RRF_K     = 60
DENSE_W   = 0.6   # higher → favour semantic similarity
SPARSE_W  = 0.4   # higher → favour exact keyword matches

print("✅ Constants loaded.")


✅ Constants loaded.


## Cell 2 — Imports & Directory Setup

In [15]:
import os
import json
import re
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi          # pip install rank-bm25

Path(TRACE_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(SUBMISSION_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Imports done. Directories ready.")


✅ Imports done. Directories ready.


## Cell 3 — Step 1: In-Memory Data Ingestion

In [16]:
# ============================================================
#  Step 1 — Load and CHUNK .md files
# ============================================================

# กำหนดขนาด Chunk (เช่น 500 ตัวอักษร, เผื่อส่วนที่ซ้อนทับกัน 50 ตัวอักษร)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    separators=["\n\n## ", "\n\n", "\n", " "]
)

documents = []

for root, dirs, files in os.walk(KNOWLEDGE_BASE_DIR):
    for fname in sorted(files):
        if fname.endswith(".md"):
            filepath = os.path.join(root, fname)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read().strip()
                
                # หั่นเนื้อหาออกเป็นชิ้นเล็กๆ
                chunks = text_splitter.split_text(content)
                
                # เก็บแต่ละ Chunk แยกเป็น 1 Document
                for i, chunk in enumerate(chunks):
                    documents.append({
                        "filename": f"{os.path.relpath(filepath, KNOWLEDGE_BASE_DIR)} (Chunk {i+1})",
                        "content": chunk
                    })
            except Exception as e:
                print(f"⚠️  Could not read {filepath}: {e}")

print(f"✅ Loaded {len(documents)} chunks into memory.")

✅ Loaded 1337 chunks into memory.


## Cell 4 — Step 2: Vectorization (No Database)

In [17]:
# ============================================================
#  Step 2 — Embed all documents (dense) + build BM25 index (sparse)
# ============================================================

print(f"🔄 Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

doc_texts = [f"Source: {doc['filename']}\n{doc['content']}" for doc in documents]

print(f"🔄 Encoding {len(doc_texts)} chunks...")
doc_embeddings = embed_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True  # unit-norm → cosine sim == dot product
)

print(f"✅ Document matrix shape: {doc_embeddings.shape}")

# ── Build BM25 sparse index ───────────────────────────────────────────────
# Tokenise by whitespace + basic Thai word splitting (split on any non-word char)
def tokenize(text: str) -> list[str]:
    """Simple whitespace + punctuation tokeniser; works for Thai & English."""
    return re.findall(r'[\w\u0E00-\u0E7F]+', text.lower())

print("🔄 Building BM25 index...")
tokenized_corpus = [tokenize(t) for t in doc_texts]
bm25_index = BM25Okapi(tokenized_corpus, k1=BM25_K1, b=BM25_B)
print(f"✅ BM25 index built over {len(tokenized_corpus)} documents.")


🔄 Loading embedding model: Qwen/Qwen3-Embedding-0.6B ...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 29687.75it/s]


🔄 Encoding 1337 chunks...


Batches: 100%|██████████| 84/84 [21:21<00:00, 15.26s/it]

✅ Document matrix shape: (1337, 1024)
🔄 Building BM25 index...
✅ BM25 index built over 1337 documents.


## Cell 5 — Step 3: Load Questions & Submission Template

In [24]:
# ============================================================
#  Step 3 — Load questions.csv and sample_submission.csv
# ============================================================

# questions.csv: [id, question, choice_1 ... choice_10]
df_questions = pd.read_csv(QUESTIONS_CSV_PATH)
CHOICE_COLS  = [f"choice_{i}" for i in range(1, 11)]

missing_q = [c for c in ["id", "question"] + CHOICE_COLS if c not in df_questions.columns]
assert not missing_q, f"Missing columns in questions.csv: {missing_q}"
print(f"✅ Questions loaded: {len(df_questions)} rows")
display(df_questions.head(3))

# sample_submission.csv: [id, answer] — used as the output template
df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert "id" in df_submission.columns and "answer" in df_submission.columns, \
    "sample_submission.csv must have columns: id, answer"
print(f"\n✅ Sample submission template loaded: {len(df_submission)} rows")
display(df_submission.head(3))

✅ Questions loaded: 100 rows


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า



✅ Sample submission template loaded: 100 rows


,id,answer
0,1,1
1,2,1
2,3,1


## Cell 6 — Helper Functions

In [56]:
# ============================================================
#  Helper Functions — Hybrid Search Edition 🚀
# ============================================================

# ── Reciprocal Rank Fusion helper ────────────────────────────────────────
def _rrf_score(rank: int, k: int = RRF_K) -> float:
    """RRF score for a document at position `rank` (1-indexed)."""
    return 1.0 / (k + rank)

def enhance_query(question: str) -> str:
    payload = {
        "model": LLM_MODEL_NAME,
        "prompt": f"""คุณคือผู้ช่วยสกัดคีย์เวิร์ด จงแปลงคำถามต่อไปนี้ให้เป็น "ข้อความค้นหาสั้นๆ" ที่มีแค่คีย์เวิร์ดสำคัญ เช่น ชื่อสินค้า รุ่น สเปค หรือหัวข้อที่ต้องการ
ห้ามอธิบาย ห้ามแต่งประโยค ให้ตอบเป็นแค่ข้อความค้นหาเดียวในบรรทัดเดียว

คำถาม: {question}
ข้อความค้นหา:""",
        "stream": False,
        "options": {
            "num_ctx": 4028,
            "num_predict": 1024,
            "temperature": 0.1,
            "top_p": 0.9,
            "repeat_penalty": 1.15
        }
    }
    try:
        resp = requests.post(OLLAMA_API_URL, json=payload, timeout=60)
        resp.raise_for_status()
        raw = resp.json().get("response", "")

        # Strip think block if the model still produces one
        clean = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE).strip()
        lines = [l.strip() for l in clean.splitlines() if l.strip()]
        enhanced = lines[0] if lines else ""
        return enhanced if enhanced else question
    except Exception as e:
        print(f"  ⚠️ Query enhancement failed, using original: {e}")
        return question


# ── Question type detector ────────────────────────────────────────────────
def detect_question_type(question: str) -> str:
    """
    Classify question into one of 6 types to guide retrieval + prompting.
    Returns: 'A', 'B', 'C', 'D', 'E', 'F'
    """
    q = question.lower()

    # E — clearly off-topic (no product/store keywords at all)
    off_topic_keywords = ["ตั๋วเครื่องบิน", "ดอกเบี้ย", "ผัดกระเพรา", "วันหยุดราชการ", "สูตร", "ราคาน้ำมัน"]
    if any(kw in q for kw in off_topic_keywords):
        return "E"

    # D — fact-checking ("เพื่อนบอกว่า...จริงไหม", "จริงหรือเปล่า")
    if re.search(r"เพื่อน(บอก|แนะนำ|ว่า)|จริงไหม|จริงหรือเปล่า|ถูกต้องไหม", q):
        return "D"

    # C — counting / budget filtering
    if re.search(r"กี่รุ่น|มีรุ่นไหนบ้าง|ราคารวม|ได้กี่|งบ\s*[฿\d]|ไม่เกิน\s*[฿\d]|มีกี่|ได้กี่ point|ได้กี่แต้ม", q):
        return "C"

    # B — comparison (X vs Y, ต่างกันยังไง)
    if re.search(r"กับ|เทียบ|ต่างกัน|เหมือนกัน|vs\b|ดีกว่า|อยู่นานกว่า|นานกว่า|ถูกกว่า|แพงกว่า", q):
        return "B"

    # F — scenario / policy (long situational questions, returns, warranty claims)
    if re.search(r"คืน|ยกเลิก|เคลม|ประกัน|trade.?in|เทิร์น|สถานะ|pre.?order|พรีออเดอร์|จัดส่ง|ส่งแบบด่วน|care\+", q):
        return "F"

    # A — direct fact lookup (default)
    return "A"


# ── Dynamic TOP_K per question type ──────────────────────────────────────
TOP_K_BY_TYPE = {
    "A": 3,   # single product, few chunks needed
    "B": 6,   # need chunks from each product being compared
    "C": 8,   # need to enumerate all matching products
    "D": 6,   # need chunks from both products
    "E": 2,   # off-topic, minimal retrieval
    "F": 5,   # policy docs + product doc
}


def hybrid_retrieve(question: str, threshold: float = 0.0, q_type: str = "A") -> list[dict]:
    N = len(documents)
    k = TOP_K_BY_TYPE.get(q_type, TOP_K_RETRIEVAL)

    enhanced_query = enhance_query(question)
    print(f"  🔍 Enhanced query: {enhanced_query}")

    q_vec         = embed_model.encode([enhanced_query], normalize_embeddings=True)
    dense_scores  = cosine_similarity(q_vec, doc_embeddings)[0]
    dense_ranks   = np.argsort(dense_scores)[::-1]

    q_tokens      = tokenize(enhanced_query)
    sparse_scores = bm25_index.get_scores(q_tokens)
    sparse_ranks  = np.argsort(sparse_scores)[::-1]

    fused = np.zeros(N, dtype=float)
    for rank, idx in enumerate(dense_ranks, start=1):
        fused[idx] += DENSE_W  * _rrf_score(rank)
    for rank, idx in enumerate(sparse_ranks, start=1):
        fused[idx] += SPARSE_W * _rrf_score(rank)

    top_indices = np.argsort(fused)[::-1][:k]

    results = []
    for i in top_indices:
        score = float(fused[i])
        if score >= threshold:
            results.append({
                "filename":     documents[i]["filename"],
                "content":      documents[i]["content"],
                "score":        score,
                "dense_score":  float(dense_scores[i]),
                "sparse_score": float(sparse_scores[i]),
            })

    if not results:
        best_idx = top_indices[0]
        results.append({
            "filename":     documents[best_idx]["filename"],
            "content":      documents[best_idx]["content"],
            "score":        float(fused[best_idx]),
            "dense_score":  float(dense_scores[best_idx]),
            "sparse_score": float(sparse_scores[best_idx]),
        })

    return results

retrieve_top_k = hybrid_retrieve


# ── Type-aware prompt builder ─────────────────────────────────────────────
def build_prompt(question: str, choices: list[str], retrieved_docs: list[dict], q_type: str = "A") -> str:

    context = "\n\n---\n\n".join(
        f"[{doc['filename']}]\n{doc['content']}" for doc in retrieved_docs
    )
    choice_lines = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))

    type_instructions = {
        "A": "ค้นหาข้อมูลที่ตรงกับคำถามในเอกสาร แล้วจับคู่กับตัวเลือก",
        "B": "อ่านข้อมูลของแต่ละสินค้าจากเอกสาร แล้วเปรียบเทียบตามที่ถาม จากนั้นเลือกตัวเลือกที่ถูกต้อง",
        "C": "ระบุรายการสินค้าทั้งหมดที่เข้าเงื่อนไขจากเอกสาร นับหรือรวมราคา แล้วจับคู่กับตัวเลือก",
        "D": "ตรวจสอบข้อเท็จจริงที่กล่าวอ้างกับข้อมูลในเอกสาร ตัดสินว่าถูกหรือผิด แล้วเลือกตัวเลือกที่ถูกต้อง",
        "E": "ถ้าเอกสารไม่มีข้อมูลที่เกี่ยวข้องกับคำถามเลย ให้เลือกตัวเลือกที่มีความหมายว่า 'ไม่มีข้อมูล' หรือ 'ไม่เกี่ยวข้อง'",
        "F": "อ่านนโยบายหรือข้อมูลสินค้าที่เกี่ยวข้องในเอกสาร แล้วตอบตามเงื่อนไขที่ระบุในคำถาม",
    }

    instruction = type_instructions.get(q_type, type_instructions["A"])

    return f"""คุณคือ AI ผู้ประเมินข้อสอบปรนัยของร้านฟ้าใหม่ (FahMai) ทำหน้าที่เลือกคำตอบที่ถูกต้องที่สุดจากตัวเลือกที่ให้มา

    [คำถาม]
    {question}

    [เอกสารอ้างอิง]
    {context}

    [ตัวเลือก]
    {choice_lines}

    [วิธีวิเคราะห์]
    {instruction}

    [กฎเหล็ก]
    - ต้องเลือกคำตอบ 1 ข้อเสมอ ห้ามไม่ตอบ
    - อ้างอิงจากเอกสารเท่านั้น ห้ามใช้ความรู้นอกเอกสาร
    - เขียน <think> สั้นๆ ก่อน ระบุหลักฐานจากเอกสาร แล้วลงท้ายด้วย \n\n"คำตอบ: X" เสมอ

    [รูปแบบที่บังคับ]
    <think>
    คำถาม: {question}
    ประเภทคำถาม: {q_type}
    หลักฐาน: [ข้อมูลที่พบในเอกสาร สั้นๆ]
    </think>
    \n\n
    คำตอบ: [เลข 1-10]"""

def call_ollama(prompt: str) -> str:
    """POST prompt to Ollama and return the raw response text."""
    payload = {
        "model": LLM_MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_ctx": 32000,
            "num_predict": 12000,
            "temperature": 0.1,
            "top_p": 0.9,
            "repeat_penalty": 1.15
        }
    }
    try:
        resp = requests.post(OLLAMA_API_URL, json=payload, timeout=600)
        resp.raise_for_status()
        raw = resp.json().get("response", "")
        return "<think>\n" + raw
    except Exception as e:
        print(f"  ⚠️ Ollama API Error: {e}")
        return ""

def extract_final_answer(raw: str) -> str:
    """Extract the text after </think> or after the word 'คำตอบ:'."""
    if not raw: return ""
    clean = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE).strip()
    m = re.search(r"คำตอบ\s*:\s*.*?(\d+)", clean)
    if m:
        return m.group(1).strip()
    lines = [l.strip() for l in clean.splitlines() if l.strip()]
    if lines:
        last_line = lines[-1]
        digit_match = re.search(r"(\d+)", last_line)
        if digit_match:
            return digit_match.group(1)
        return last_line
    return clean


def append_trace(record: dict):
    """Append one JSON record to the trace log (JSONL format)."""
    with open(TRACE_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def clear_trace():
    """Clear the trace log by truncating the file."""
    if os.path.exists(TRACE_LOG_PATH):
        with open(TRACE_LOG_PATH, "w", encoding="utf-8"):
            pass


print("✅ Hybrid-search helper functions loaded.")


✅ Hybrid-search helper functions loaded.


## Cell 7 — Steps 4–6: Main Pipeline Loop

In [57]:
# ============================================================
#  Steps 4-6 — Retrieval → LLM → Parse → Fill submission
# ============================================================

# Work on a copy of the template; iterate by id order from sample_submission
df_result = df_submission.copy()
df_result["answer"] = df_result["answer"].astype(object) 
q_lookup = df_questions.set_index("id")

clear_trace()

for idx, row in df_result.iterrows():
    q_id = int(row["id"])

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(df_result)}] ID={q_id}")

    try:
        q_row    = q_lookup.loc[q_id]
        question = str(q_row["question"])
        choices  = [str(q_row[c]) for c in CHOICE_COLS]

        print(f"  ❓ {question[:80]}{'...' if len(question) > 80 else ''}")

        # --- Step 4: Retrieve ---
        retrieved           = retrieve_top_k(question)
        retrieved_filenames = [r["filename"] for r in retrieved]
        print(f"  📄 Retrieved: {retrieved_filenames}")

        # --- Step 5: LLM Generation ---
        prompt   = build_prompt(question, choices, retrieved)
        raw_resp = call_ollama(prompt)

        # --- Step 6: Parse ---
        raw_answer = extract_final_answer(raw_resp)

        match_index = None
        is_fallback = False
        clean_raw = str(raw_answer).strip()

        # หาตัวเลข 1-10 จากผลลัพธ์ที่ extract มาได้
        numbers = re.findall(r'\b([1-9]|10)\b', clean_raw) 
        if numbers:
            match_index = int(numbers[0])
        else:
            match_index = 9 # Absolute fallback
            is_fallback = True

        # --- Explicit Print Logging ---
        if is_fallback:
            print(f"  ⚠️ FALLBACK (Defaulting to 9). Raw: '{str(raw_answer).strip()[:40]}...'")
        else:
            print(f"  🤖 Extracted Answer: {match_index}")

        # ---> รวบ append_trace ไว้ที่เดียว (1 ข้อ = 1 บรรทัดใน Log)
        append_trace({
            "question_id":     q_id,
            "question":        question,
            "prompt":          prompt,
            "retrieved_files": retrieved_filenames,
            "raw_response":    raw_resp,
            "final_answer":    match_index,
            "is_fallback":     is_fallback 
        })

        # Write the NUMBER into the submission DataFrame
        df_result.at[idx, "answer"] = int(match_index)

    except Exception as e:
        err_msg = str(e)
        print(f"  ❌ ERROR: {err_msg}")
        
        # กรณีเกิด Error ก็ Log แบบรวบยอดเช่นกัน
        append_trace({
            "question_id": q_id, 
            "error": err_msg,
            "is_fallback": True
        })

# --- Save timestamped submission CSV ---
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(SUBMISSION_DIR, f"submission_{timestamp}.csv")
df_result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Submission saved → {output_path}")
display(df_result)


[1/100] ID=1
  ❓ Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  🔍 Enhanced query: Watch S3 Ultra ATM
  📄 Retrieved: ['products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 15)', 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 1)', 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md (Chunk 3)']
  🤖 Extracted Answer: 5

[2/100] ID=2
  ❓ ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ
  🔍 Enhanced query: SaiFah Pen Gen 2
  📄 Retrieved: ['products/SF-TB-002_saifah_tab_s9.md (Chunk 13)', 'products/SF-TB-005_saifah_tab_mini_7.md (Chunk 12)', 'products/SF-TB-001_saifah_tab_s9_pro.md (Chunk 17)']
  🤖 Extracted Answer: 7

[3/100] ID=3
  ❓ หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ
  🔍 Enhanced query: HeadPro X1 Bluetooth version
  📄 Retrieved: ['products/KS-HP-001_kluensiang_headpro_x1.md (Chunk 12)', 'products/KS-HP-001_kluensiang_headpro_x1.md (Chunk 1)', 'products/KS-HP-002_kluensiang_headpro_x1_se.md (Chunk 9)']
  🤖 Extracted Answer: 2

[4/100] ID=4
  ❓ อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเ

,id,answer
0,1,5
1,2,7
2,3,2
3,4,6
4,5,9
...,...,...
95,96,5
96,97,7
97,98,2
98,99,6


## Cell 8 — Sanity Check

In [43]:
# ============================================================
#  Sanity Check — submission stats & trace entries
# ============================================================

print(f"Total rows         : {len(df_result)}")
print(f"Answered (non-null): {df_result['answer'].notna().sum()}")
print(f"Still template val : {(df_result['answer'] == df_submission['answer']).sum()}")
print()
display(df_result)

with open(TRACE_LOG_PATH, "r", encoding="utf-8") as f:
    trace_lines = f.readlines()
print(f"\nTrace log entries: {len(trace_lines)}")

Total rows         : 100
Answered (non-null): 100
Still template val : 13



,id,answer
0,1,5
1,2,7
2,3,2
3,4,6
4,5,9
...,...,...
95,96,1
96,97,9
97,98,4
98,99,6



Trace log entries: 100
